# 02a — Rezagos e índice

Bloques 1–2 del cuaderno 02: rezagos (solo memoria) e índice meteorológico. Escribe `expanded/firms_spain_final_lags_index.csv`.

# 1 — Variables de rezago para `wildfire_data_cleaned.csv`

Este notebook crea variables meteorológicas retrospectivas para cada observación. Agrega las mediciones disponibles por **comunidad autónoma y fecha** y calcula los rezagos sobre días naturales. Así, los periodos previos excluyen el día de la observación.

**Importante:** el CSV contiene observaciones de incendios, no una tabla meteorológica diaria completa e independiente. Los promedios diarios regionales se calculan a partir de las filas disponibles del propio CSV. Los valores deben interpretarse como aproximaciones regionales basadas en esas observaciones; no como clima exacto en cada coordenada. Las ventanas se dejan como `NaN` si falta cualquiera de sus días.

## Imports (celda única del cuaderno)

In [ ]:
# Imports del cuaderno (celda única) — todas las dependencias aquí; las demás celdas no importan nada.
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from wildfire.data.io import load_csv, save_csv


### 1.1 — Carga y fechas

Carga el CSV limpio y normaliza `acq_date`; punto de partida del bloque.

In [ ]:

df = load_csv("data/processed/expanded/wildfire_data_cleaned.csv")

df['acq_date'] = pd.to_datetime(df['acq_date'], errors='coerce').dt.normalize()
required = {'acq_date', 'ccaa', 'precipitation', 'temperature_2m'}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f'Faltan columnas necesarias: {sorted(missing)}')

print(f'{len(df):,} filas; fechas {df.acq_date.min().date()} a {df.acq_date.max().date()}')
print(f"Comunidades autónomas: {df['ccaa'].nunique(dropna=True)}")
df.head()


Lo que ves: cuántas detecciones hay en total, entre qué fechas van y en cuántas comunidades aparecen. Es el punto de partida: todo lo que sigue (rezagos, índice) se calcula sobre estas filas y estas fechas. Primera lectura: dos años completos repartidos por comunidades autónomas.

## Definición de las variables

Se calculan precipitación acumulada en los 3, 7 y 15 días anteriores; temperatura media en los 3, 7 y 15 días anteriores; y rezagos de un día para otras variables meteorológicas disponibles. Las ventanas usan días naturales y son estrictas: requieren datos diarios para todo el periodo.

### 1.2 — Medias diarias y rezagos

Agrega por CCAA×día y calcula `lag_1d`, sumas de precipitación y medias de temperatura a 3/7/15 días; produce `df_lags` (solo memoria).

In [ ]:
# Variables meteorológicas disponibles en el archivo.
weather_cols = [c for c in [
    'precipitation', 'rain', 'temperature_2m', 'relative_humidity_2m',
    'wind_speed_10m', 'wind_gusts_10m', 'vapour_pressure_deficit',
    'surface_pressure', 'soil_moisture_0_to_7cm', 'soil_moisture_7_to_28cm',
    'shortwave_radiation', 'et0_fao_evapotranspiration', 'boundary_layer_height'
] if c in df.columns]

# Para evitar que el texto u otros valores no numéricos contaminen los promedios.
for col in weather_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Una media por comunidad y día a partir de las filas disponibles.
daily = (df.dropna(subset=['acq_date', 'ccaa'])
         .groupby(['ccaa', 'acq_date'], as_index=False)[weather_cols]
         .mean())

date_min, date_max = df['acq_date'].min(), df['acq_date'].max()
feature_frames = []
for region, part in daily.groupby('ccaa', sort=False):
    part = part.set_index('acq_date').sort_index()
    # Reindexado a días naturales: los días sin observación permanecen ausentes.
    calendar = pd.date_range(date_min, date_max, freq='D')
    part = part.reindex(calendar)
    part.index.name = 'acq_date'
    features = pd.DataFrame(index=part.index)
    features['ccaa'] = region

    # Valor del día inmediatamente anterior (día t-1).
    for col in weather_cols:
        features[f'{col}_lag_1d'] = part[col].shift(1)

    # Precipitación acumulada en los N días anteriores, sin incluir el día t.
    if 'precipitation' in part:
        for n in (3, 7, 15):
            features[f'precipitation_sum_prev_{n}d'] = part['precipitation'].shift(1).rolling(n, min_periods=n).sum()
    # Temperatura media en los N días anteriores, sin incluir el día t.
    if 'temperature_2m' in part:
        for n in (3, 7, 15):
            features[f'temperature_2m_mean_prev_{n}d'] = part['temperature_2m'].shift(1).rolling(n, min_periods=n).mean()
    feature_frames.append(features.reset_index())

lag_features = pd.concat(feature_frames, ignore_index=True)
lag_cols = [c for c in lag_features.columns if c not in ('ccaa', 'acq_date')]
df_lags = df.merge(lag_features, on=['ccaa', 'acq_date'], how='left', validate='many_to_one')
print(f'Variables nuevas: {len(lag_cols)}')
df_lags[['acq_date', 'ccaa', 'precipitation', 'temperature_2m'] + lag_cols[:8]].head(10)

Lo que ves: cuántas variables nuevas de 'ayer' se crean (rezago de 1 día + medias y sumas de 3/7/15 días) y una vista previa de 10 filas. Las celdas vacías (NaN) del principio son normales: el 1 de enero no tiene 'ayer'. El índice final apenas usa estas columnas (demasiados huecos), pero aquí quedan construidas y medidas.

## Revisar cobertura y guardar

Las fechas iniciales y los periodos con días regionales ausentes tendrán valores nulos. Revise la cobertura antes de usar estas variables en un modelo.

**Aviso:** los rezagos no se exportan a CSV a propósito — el reindexado a días naturales más la exigencia de ventanas completas dejan una cobertura de NaN muy alta. `df_lags` vive solo en memoria y se revisitará más adelante.

### 1.3 — Cobertura de NaNs

Informa la cobertura de cada rezago; inicios y huecos regionales quedan como NaN por diseño.

In [ ]:
coverage = df_lags[lag_cols].notna().mean().sort_values(ascending=False).rename('proporcion_no_nula')
display(coverage.to_frame())

print(df_lags.isna().sum())

Lo que ves: para cada variable de rezago, qué proporción de filas tiene valor (1,0 = completa) y el conteo de huecos. Lectura: el rezago de 1 día cubre ~83 %, pero las ventanas de 7/15 días caen mucho — solo existen donde hubo observaciones regionales seguidas. Conclusión: los rezagos largos no sirven como entrada general; por eso ni el índice ni el modelado dependen de ellos.

# 2 — Índice meteorológico compuesto de peligro de incendio

Este notebook combina las variables meteorológicas presentes en `firms_spain_final.csv` en un **índice relativo de 0 a 100**. Se usan temperatura, humedad relativa, viento, precipitación, déficit de presión de vapor (VPD) y humedad del suelo.

## Interpretación y límites

Es un índice exploratorio construido para este archivo, **no es el Fire Weather Index (FWI) oficial ni una predicción validada**. Las variables se convierten en percentiles dentro de las filas del CSV; por ello, el resultado indica posición relativa frente a las observaciones de este conjunto, no un nivel absoluto de peligro. Además, el CSV contiene localizaciones con detecciones de incendios, así que no representa todas las condiciones meteorológicas de España. No debe usarse por sí solo para alertas ni decisiones operativas.

Las ponderaciones iniciales son explícitas y editables: temperatura 20%, humedad relativa 20%, viento 20%, precipitación 15%, VPD 15% y humedad del suelo 10%. Los valores altos de temperatura, viento y VPD elevan el score; valores altos de humedad relativa, precipitación y humedad del suelo lo reducen. Se requiere un mínimo de cuatro componentes disponibles por fila; el score ponderado se ajusta a los componentes presentes.

### 2.1 — Carga y requisitos

Carga el CSV limpio y exige las 6 variables del índice.

In [ ]:

df = load_csv("data/processed/expanded/wildfire_data_cleaned.csv")
required = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'precipitation', 'vapour_pressure_deficit', 'soil_moisture_0_to_7cm']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Faltan variables necesarias: {missing}')
for col in required:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'{len(df):,} observaciones')
print('Periodo:', df['acq_date'].min(), 'a', df['acq_date'].max())
df[required].describe().T


Lo que ves: cuántas observaciones alimentan el índice y su periodo. Misma base que en 1.1, ahora como punto de partida del score meteorológico.

## Construcción del score

Cada variable se transforma a percentil empírico entre 0 y 1. Para humedad relativa, precipitación y humedad del suelo se invierte el percentil porque valores altos reducen el peligro meteorológico. La precipitación es la medición disponible en la fecha de la observación; el archivo no aporta una serie diaria completa para calcular acumulados antecedentes fiables.

### 2.2 — Score ponderado

Percentiles por variable y media ponderada 20/20/20/15/15/10 con mínimo de 4 componentes; produce `indice_meteorologico`.

In [ ]:
# peso y sentido del efecto en el peligro (True = un valor alto aumenta el score)
components = {
    'temperature_2m': {'weight': 0.20, 'higher_is_riskier': True},
    'relative_humidity_2m': {'weight': 0.20, 'higher_is_riskier': False},
    'wind_speed_10m': {'weight': 0.20, 'higher_is_riskier': True},
    'precipitation': {'weight': 0.15, 'higher_is_riskier': False},
    'vapour_pressure_deficit': {'weight': 0.15, 'higher_is_riskier': True},
    'soil_moisture_0_to_7cm': {'weight': 0.10, 'higher_is_riskier': False},
}
assert np.isclose(sum(v['weight'] for v in components.values()), 1.0)

risk_subscores = pd.DataFrame(index=df.index)
for col, spec in components.items():
    percentile = df[col].rank(method='average', pct=True)
    risk_subscores[col + '_risk_pct'] = percentile if spec['higher_is_riskier'] else 1 - percentile

weights = pd.Series({col + '_risk_pct': spec['weight'] for col, spec in components.items()})
available_weight = risk_subscores.notna().mul(weights).sum(axis=1)
weighted_score = risk_subscores.mul(weights, axis='columns').sum(axis=1, min_count=1) / available_weight
components_available = risk_subscores.notna().sum(axis=1)
df['indice_meteorologico'] = (weighted_score * 100).where(components_available >= 4)
df['componentes_disponibles'] = components_available
df['nivel_relativo'] = pd.cut(df['indice_meteorologico'], bins=[-0.001, 100/3, 200/3, 100], labels=['Bajo', 'Medio', 'Alto'], include_lowest=True)

print('Cobertura del índice:', f"{df['indice_meteorologico'].notna().mean():.1%}")
df[['indice_meteorologico', 'componentes_disponibles', 'nivel_relativo']].describe(include='all')

Lo que ves: qué % de filas tiene índice calculado (debería rondar el 100 %) y el resumen del score con sus tramos Bajo/Medio/Alto. Ojo: el índice es relativo de 0 a 100 — 50 no significa 'peligro medio' en absoluto, sino 'la mediana de estos dos años'.

## Inspección y exportación

Los rangos Bajo/Medio/Alto son tercios fijos de la escala 0–100, no umbrales oficiales. Para un análisis relativo a la muestra también conviene revisar los cuantiles y la distribución.

### 2.3 — Inspección y guardado

Histograma, resumen por CCAA y guardado en `firms_spain_final_lags_index.csv`.

In [ ]:
display(df['indice_meteorologico'].describe(percentiles=[.1, .25, .5, .75, .9]).to_frame())
ax = df['indice_meteorologico'].dropna().plot(kind='hist', bins=25, figsize=(8, 4), color='#d97706', edgecolor='white')
ax.set(title='Distribución del índice meteorológico compuesto', xlabel='Índice relativo (0–100)', ylabel='Número de observaciones')
plt.tight_layout()
plt.show()

if 'ccaa' in df.columns:
    resumen_ccaa = (df.groupby('ccaa', dropna=False)
                    .agg(observaciones=('indice_meteorologico', 'count'),
                         indice_medio=('indice_meteorologico', 'mean'),
                         indice_mediano=('indice_meteorologico', 'median'))
                    .sort_values('indice_medio', ascending=False))
    display(resumen_ccaa)

output_path = 'data/processed/expanded/firms_spain_final_lags_index.csv'
save_csv(df, output_path)
print(f'CSV enriquecido guardado en: {output_path}')

Lo que ves: el resumen del índice (deciles incluidos), su histograma y el ranking por comunidad (observaciones, media y mediana, ordenado por media). El histograma muestra dónde se acumula el índice: la mayoría de detecciones cae en valores medios-bajos, con colas en ambos extremos. La tabla ordena comunidades por índice medio: Madrid y Baleares arriba no significa que ardan más, sino que cuando hay detección las condiciones suelen ser más severas; Asturias abajo no es 'seguro', es que muchas de sus detecciones son quemas invernales suaves. Moraleja: el índice describe las condiciones del día con fuego, no la probabilidad de fuego.